In [1]:
%%writefile attention.py
import math
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float=0.1, attention_scale: float=None, use_rope: bool=True, rope_base: float=10000.0, causal: bool=False, cross: bool=False):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be evenly divisible by n_heads'
        assert not (causal and cross), 'cross-attention is never causal'
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.causal = causal
        self.cross = cross
        self.attention_scale = 1.0 / math.sqrt(self.d_head) if attention_scale is None else float(attention_scale)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.use_rope = use_rope and (not cross)
        if self.use_rope:
            assert self.d_head % 2 == 0, 'RoPE needs an even head dimension'
            inv_freq = 1.0 / rope_base ** (torch.arange(0, self.d_head, 2, dtype=torch.float32) / self.d_head)
            angles = torch.arange(max_seq_len, dtype=torch.float32).unsqueeze(1) * inv_freq.unsqueeze(0)
            self.register_buffer('rope_cos', torch.cos(angles), persistent=False)
            self.register_buffer('rope_sin', torch.sin(angles), persistent=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            self.register_buffer('causal_mask', torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)), persistent=False)

    def _apply_rope(self, x: torch.Tensor, start: int) -> torch.Tensor:
        T = x.size(2)
        cos = self.rope_cos[start:start + T].to(torch.float32)
        sin = self.rope_sin[start:start + T].to(torch.float32)
        x1, x2 = (x[..., :self.d_head // 2].float(), x[..., self.d_head // 2:].float())
        out = torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)
        return out.to(x.dtype)

    def _split_heads(self, t: torch.Tensor) -> torch.Tensor:
        B, L, _ = t.shape
        return t.view(B, L, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x: torch.Tensor, memory: Optional[torch.Tensor]=None, past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]]=None, use_cache: bool=False, key_padding_mask: Optional[torch.Tensor]=None):
        B, T, C = x.shape
        q = self._split_heads(self.q_proj(x))
        if self.cross:
            if past_kv is not None:
                k, v = past_kv
            else:
                assert memory is not None, 'cross-attention needs encoder output (memory)'
                k = self._split_heads(self.k_proj(memory))
                v = self._split_heads(self.v_proj(memory))
            past_len = 0
            total_len = k.size(2)
        else:
            k = self._split_heads(self.k_proj(x))
            v = self._split_heads(self.v_proj(x))
            past_len = 0
            if past_kv is not None:
                past_len = past_kv[0].size(2)
            if self.use_rope:
                q = self._apply_rope(q, past_len)
                k = self._apply_rope(k, past_len)
            if past_kv is not None:
                past_k, past_v = past_kv
                k = torch.cat([past_k, k], dim=2)
                v = torch.cat([past_v, v], dim=2)
            total_len = past_len + T
        present = (k, v) if use_cache else None
        attn_scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = None
        if self.causal:
            mask = self.causal_mask[past_len:total_len, :total_len]
        if key_padding_mask is not None:
            kpm = key_padding_mask.to(torch.bool).unsqueeze(1)
            mask = kpm if mask is None else mask.unsqueeze(0) & kpm
            mask = mask | ~mask.any(dim=-1, keepdim=True)
            mask = mask.unsqueeze(1)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(~mask, float('-inf'))
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)
        attn_output = attn_probs @ v
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)
        attn_output = self.out_proj(attn_output)
        attn_output = self.resid_dropout(attn_output)
        if use_cache:
            return (attn_output, present)
        return attn_output


Writing attention.py


In [2]:
%%writefile model_arch.py
import math
from dataclasses import dataclass
from typing import Optional, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.utils import ModelOutput
from attention import MultiHeadAttention

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 8
    n_heads: int = 8
    d_ff: int = 2240
    max_seq_len: int = 1024
    dropout: float = 0
    pos_emb_type: str = 'rope'
    attention_scale: float = None
    norm_first: bool = True
    bos_token_id: int = None
    eos_token_id: int = None
    n_enc_layers: int = None
    n_dec_layers: int = None

    def __post_init__(self):
        if self.n_enc_layers is None:
            self.n_enc_layers = max(1, self.n_layers // 2)
        if self.n_dec_layers is None:
            self.n_dec_layers = max(1, self.n_layers - self.n_layers // 2)

@dataclass
class Seq2SeqLMOutput(ModelOutput):
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[List[Tuple[Tuple[torch.Tensor, torch.Tensor], Tuple[torch.Tensor, torch.Tensor]]]] = None

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model, dtype=torch.float32)
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    return pe

class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class EncoderBlock(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = getattr(cfg, 'norm_first', True)
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(d_model=cfg.d_model, n_heads=cfg.n_heads, max_seq_len=cfg.max_seq_len, dropout=cfg.dropout, attention_scale=cfg.attention_scale, use_rope=cfg.pos_emb_type == 'rope', causal=False)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, key_padding_mask=None):
        if self.norm_first:
            x = x + self.attn(self.ln1(x), key_padding_mask=key_padding_mask)
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x, key_padding_mask=key_padding_mask))
            x = self.ln2(x + self.ffn(x))
        return x

class DecoderBlock(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = getattr(cfg, 'norm_first', True)
        use_rope = cfg.pos_emb_type == 'rope'
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = MultiHeadAttention(d_model=cfg.d_model, n_heads=cfg.n_heads, max_seq_len=cfg.max_seq_len, dropout=cfg.dropout, attention_scale=cfg.attention_scale, use_rope=use_rope, causal=True)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = MultiHeadAttention(d_model=cfg.d_model, n_heads=cfg.n_heads, max_seq_len=cfg.max_seq_len, dropout=cfg.dropout, attention_scale=cfg.attention_scale, use_rope=False, cross=True)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, memory: torch.Tensor, past_kv=None, use_cache: bool=False, memory_mask=None):
        self_past, cross_past = (None, None) if past_kv is None else past_kv
        self_present = cross_present = None
        if self.norm_first:
            a = self.self_attn(self.ln1(x), past_kv=self_past, use_cache=use_cache)
            if use_cache:
                a, self_present = a
            x = x + a
            c = self.cross_attn(self.ln2(x), memory=memory, past_kv=cross_past, use_cache=use_cache, key_padding_mask=memory_mask)
            if use_cache:
                c, cross_present = c
            x = x + c
            x = x + self.ffn(self.ln3(x))
        else:
            a = self.self_attn(x, past_kv=self_past, use_cache=use_cache)
            if use_cache:
                a, self_present = a
            x = self.ln1(x + a)
            c = self.cross_attn(x, memory=memory, past_kv=cross_past, use_cache=use_cache, key_padding_mask=memory_mask)
            if use_cache:
                c, cross_present = c
            x = self.ln2(x + c)
            x = self.ln3(x + self.ffn(x))
        if use_cache:
            return (x, (self_present, cross_present))
        return x

class EncoderDecoderLM(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == 'sinusoidal':
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer('pos_emb', pe, persistent=False)
        self.enc_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_enc_layers)])
        self.enc_ln_f = nn.LayerNorm(cfg.d_model)
        self.dec_blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_dec_layers)])
        self.dec_ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)

    @property
    def decoder_start_token_id(self) -> int:
        return self.cfg.bos_token_id if self.cfg.bos_token_id is not None else 0

    def shift_right(self, labels: torch.Tensor) -> torch.Tensor:
        start = torch.full_like(labels[:, :1], self.decoder_start_token_id)
        dec = torch.cat([start, labels[:, :-1]], dim=1)
        return dec.masked_fill(dec == -100, self.decoder_start_token_id)

    def encode(self, src_ids: torch.Tensor, attention_mask: Optional[torch.Tensor]=None) -> torch.Tensor:
        B, S = src_ids.shape
        assert S <= self.cfg.max_seq_len, f'Source length {S} exceeds max sequence length {self.cfg.max_seq_len}'
        x = self.tok_emb(src_ids)
        if self.cfg.pos_emb_type == 'sinusoidal':
            if attention_mask is None:
                x = x + self.pos_emb[:S]
            else:
                pos_ids = (attention_mask.long().cumsum(dim=-1) - 1).clamp(min=0)
                x = x + self.pos_emb[pos_ids]
        key_padding_mask = None if attention_mask is None else attention_mask.to(torch.bool)
        for block in self.enc_blocks:
            x = block(x, key_padding_mask=key_padding_mask)
        return self.enc_ln_f(x)

    def forward(self, input_ids: torch.Tensor=None, labels: torch.Tensor=None, idx: torch.Tensor=None, decoder_input_ids: torch.Tensor=None, encoder_outputs: Optional[torch.Tensor]=None, past_key_values: Optional[List]=None, use_cache: bool=False, attention_mask: Optional[torch.Tensor]=None, last_token_only: bool=False):
        if idx is None:
            idx = input_ids
        if encoder_outputs is None:
            assert idx is not None, 'Either input_ids / idx or encoder_outputs must be provided'
            encoder_outputs = self.encode(idx, attention_mask)
        if decoder_input_ids is None:
            assert labels is not None, 'Either decoder_input_ids or labels must be provided'
            decoder_input_ids = self.shift_right(labels)
        B, T = decoder_input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0][0].size(2)
        assert past_len + T <= self.cfg.max_seq_len, f'Decoder length {past_len + T} exceeds max sequence length {self.cfg.max_seq_len}'
        memory_mask = None if attention_mask is None else attention_mask.to(torch.bool)
        x = self.tok_emb(decoder_input_ids)
        if self.cfg.pos_emb_type == 'sinusoidal':
            x = x + self.pos_emb[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, block in enumerate(self.dec_blocks):
            past_kv = None if past_key_values is None else past_key_values[i]
            if use_cache:
                x, present = block(x, encoder_outputs, past_kv=past_kv, use_cache=True, memory_mask=memory_mask)
                presents.append(present)
            else:
                x = block(x, encoder_outputs, past_kv=past_kv, memory_mask=memory_mask)
        if last_token_only:
            x = x[:, -1:, :]
        x = self.dec_ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.contiguous().view(-1, self.cfg.vocab_size), labels.contiguous().view(-1), ignore_index=-100)
            return Seq2SeqLMOutput(loss=loss, logits=logits, past_key_values=presents)
        if use_cache:
            return (logits, presents)
        return logits

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_p: float=1.0, eos_token_id: int=None, attention_mask: torch.Tensor=None, stop_check=None, check_every: int=16) -> torch.Tensor:
        was_training = self.training
        self.eval()
        B = idx.size(0)
        device = idx.device
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=device, dtype=torch.bool)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        max_new_tokens = min(max_new_tokens, self.cfg.max_seq_len)
        memory = self.encode(idx, attention_mask)
        cur = torch.full((B, 1), self.decoder_start_token_id, dtype=torch.long, device=device)
        generated = torch.empty(B, 0, dtype=torch.long, device=device)
        past_key_values = None
        for step in range(max_new_tokens):
            logits, past_key_values = self(None, encoder_outputs=memory, decoder_input_ids=cur, past_key_values=past_key_values, use_cache=True, attention_mask=attention_mask, last_token_only=True)
            logits = logits[:, -1, :] / max(temperature, 1e-05)
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, float('-inf'))
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            if eos_token_id is not None:
                idx_next = torch.where(finished.unsqueeze(1), torch.full_like(idx_next, eos_token_id), idx_next)
                finished = finished | (idx_next.squeeze(1) == eos_token_id)
            generated = torch.cat((generated, idx_next), dim=1)
            cur = idx_next
            if eos_token_id is not None and finished.all():
                break
            if stop_check is not None and (step + 1) % check_every == 0:
                satisfied = torch.as_tensor(stop_check(generated), dtype=torch.bool, device=device)
                if (satisfied | finished).all():
                    break
        if was_training:
            self.train()
        return torch.cat((idx, generated), dim=1)

    @staticmethod
    def _repeated_ngram_tokens(seq, n):
        if len(seq) < n - 1:
            return []
        prefix = tuple(seq[len(seq) - (n - 1):])
        banned = []
        for i in range(len(seq) - n + 1):
            if tuple(seq[i:i + n - 1]) == prefix:
                banned.append(seq[i + n - 1])
        return banned

    @torch.no_grad()
    def generate_beam(self, idx: torch.Tensor, max_new_tokens: int, num_beams: int=4, temperature: float=1.0, attention_mask: torch.Tensor=None, no_repeat_ngram_size: int=0, banned_token_ids=None, stop_check=None, check_every: int=16) -> torch.Tensor:
        was_training = self.training
        self.eval()
        B, T = idx.shape
        nb = num_beams
        device = idx.device
        max_new_tokens = min(max_new_tokens, self.cfg.max_seq_len)
        if attention_mask is None:
            attention_mask = torch.ones(B, T, dtype=torch.bool, device=device)
        attention_mask = attention_mask.to(device=device, dtype=torch.bool)
        memory = self.encode(idx, attention_mask).repeat_interleave(nb, dim=0)
        mask = attention_mask.repeat_interleave(nb, dim=0)
        cur = torch.full((B * nb, 1), self.decoder_start_token_id, dtype=torch.long, device=device)
        past = None
        beam_scores = torch.zeros(B, nb, device=device)
        beam_scores[:, 1:] = float('-inf')
        gen = torch.empty(B * nb, 0, dtype=torch.long, device=device)
        row_offset = (torch.arange(B, device=device) * nb).unsqueeze(1)
        for step in range(max_new_tokens):
            out, past = self(None, encoder_outputs=memory, decoder_input_ids=cur, past_key_values=past, use_cache=True, attention_mask=mask, last_token_only=True)
            next_logits = out[:, -1, :]
            log_probs = F.log_softmax(next_logits.float() / max(temperature, 1e-05), dim=-1)
            if banned_token_ids:
                log_probs[:, banned_token_ids] = float('-inf')
            if no_repeat_ngram_size > 0 and gen.size(1) >= no_repeat_ngram_size - 1:
                for r, seq in enumerate(gen.tolist()):
                    bad = self._repeated_ngram_tokens(seq, no_repeat_ngram_size)
                    if bad:
                        log_probs[r, bad] = float('-inf')
            vocab = log_probs.size(-1)
            scores = (beam_scores.view(-1, 1) + log_probs).reshape(B, nb * vocab)
            beam_scores, top_idx = scores.topk(nb, dim=-1)
            src_beam = (top_idx // vocab + row_offset).view(-1)
            next_tok = (top_idx % vocab).view(-1, 1)
            gen = torch.cat([gen[src_beam], next_tok], dim=1)
            past = [((sk[src_beam], sv[src_beam]), cross_kv) for (sk, sv), cross_kv in past]
            cur = next_tok
            if stop_check is not None and (step + 1) % check_every == 0:
                best_so_far = gen.view(B, nb, -1)[:, 0]
                satisfied = torch.as_tensor(stop_check(best_so_far), dtype=torch.bool)
                if satisfied.all():
                    break
        best = gen.view(B, nb, -1)[:, 0]
        if was_training:
            self.train()
        return torch.cat([idx, best], dim=1)


Writing model_arch.py


In [3]:
import os
import json
from transformers import PreTrainedTokenizer
from huggingface_hub import hf_hub_download
from transformers import PreTrainedTokenizer

class MyCustomTokenizer(PreTrainedTokenizer):
    END_OF_WORD = '</w>'

    def __init__(self, vocab_file, merges_file, pad_token='<pad>', unk_token='<unk>', bos_token='<s>', eos_token='</s>', **kwargs):
        with open(vocab_file, 'r', encoding='utf-8') as f:
            self.vocab = json.load(f)
        self.id_to_token = {i: t for t, i in self.vocab.items()}
        self.byte_to_char = self._build_byte_to_char()
        self.char_to_byte = {c: b for b, c in self.byte_to_char.items()}
        self.merge_ranks = self._load_merges(merges_file)
        self.word_cache = {}
        kwargs.setdefault('clean_up_tokenization_spaces', False)
        super().__init__(pad_token=pad_token, unk_token=unk_token, bos_token=bos_token, eos_token=eos_token, **kwargs)

    @staticmethod
    def _build_byte_to_char():
        printable = list(range(ord('!'), ord('~') + 1)) + list(range(ord('¡'), ord('¬') + 1)) + list(range(ord('®'), ord('ÿ') + 1))
        byte_values = printable[:]
        char_codes = printable[:]
        extra = 0
        for b in range(256):
            if b not in printable:
                byte_values.append(b)
                char_codes.append(256 + extra)
                extra += 1
        return {b: chr(c) for b, c in zip(byte_values, char_codes)}

    def _byte_ids_to_symbol(self, text):
        symbol = ''
        for part in text.split('+'):
            if part == self.END_OF_WORD:
                symbol += part
            else:
                symbol += self.byte_to_char[int(part[2:])]
        return symbol

    def _load_merges(self, merges_file):
        pairs = []
        with open(merges_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#version'):
                    continue
                parts = line.split(' ')
                if len(parts) == 2:
                    pairs.append((parts[0], parts[1]))
        written_as_byte_ids = len(pairs) > 0 and pairs[0][0].startswith('b:')
        merge_ranks = {}
        for left, right in pairs:
            if written_as_byte_ids:
                left = self._byte_ids_to_symbol(left)
                right = self._byte_ids_to_symbol(right)
            merge_ranks[left, right] = len(merge_ranks)
        return merge_ranks

    @property
    def vocab_size(self):
        return len(self.vocab)

    def get_vocab(self):
        return self.vocab

    def _split_word(self, word):
        if word in self.word_cache:
            return self.word_cache[word]
        symbols = [self.byte_to_char[b] for b in word.encode('utf-8')] + [self.END_OF_WORD]
        while len(symbols) > 1:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            known_pairs = [p for p in pairs if p in self.merge_ranks]
            if not known_pairs:
                break
            best_pair = min(known_pairs, key=lambda p: self.merge_ranks[p])
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    new_symbols.append(best_pair[0] + best_pair[1])
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
        tokens = [s if s in self.vocab else self.unk_token for s in symbols]
        if len(self.word_cache) > 500000:
            self.word_cache.clear()
        self.word_cache[word] = tokens
        return tokens

    def _tokenize(self, text):
        tokens = []
        for word in text.split():
            tokens.extend(self._split_word(word))
        return tokens

    def convert_tokens_to_string(self, tokens):
        byte_list = []
        for token in tokens:
            if token in self.all_special_tokens:
                continue
            word_ends = token.endswith(self.END_OF_WORD)
            if word_ends:
                token = token[:-len(self.END_OF_WORD)]
            byte_list.extend((self.char_to_byte[c] for c in token if c in self.char_to_byte))
            if word_ends:
                byte_list.append(ord(' '))
        return bytes(byte_list).decode('utf-8', errors='replace').strip()

    def _convert_token_to_id(self, token):
        return self.vocab.get(token, self.vocab.get(self.unk_token, 1))

    def _convert_id_to_token(self, index):
        return self.id_to_token.get(index, self.unk_token)

    def save_vocabulary(self, save_directory, filename_prefix=None):
        prefix = filename_prefix + '-' if filename_prefix else ''
        vocab_file = os.path.join(save_directory, prefix + 'vocab.json')
        merges_file = os.path.join(save_directory, prefix + 'merges.txt')
        with open(vocab_file, 'w', encoding='utf-8') as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)
        with open(merges_file, 'w', encoding='utf-8') as f:
            f.write('#version: 0.2\n')
            for (left, right), _ in sorted(self.merge_ranks.items(), key=lambda item: item[1]):
                f.write(f'{left} {right}\n')
        return (vocab_file, merges_file)


In [4]:
import os
import re
import json
import math
import time
import random
import zipfile
import copy
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, login

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

hf_token = hf_token if (hf_token and str(hf_token).strip()) else None
if hf_token:
    login(token=hf_token)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("running on:", device)


running on: cuda


In [5]:
seed = 42
debug_mode = False

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

dataset_repo = "Exploration-Lab/CS-781-Capstone"
base_model_repo = "Granth2/cs781-phase1-Submission-8"
tokenizer_repo = "Granth2/cs781-custom-bpe-30k-Version-10-40-40-10"
summarizer_repo = None

context_length = 1024
hidden_size = 768
num_layers = 12
num_heads = 12

output_dir = "/kaggle/working/cs781_phase7"
pred_file = os.path.join(output_dir, "prediction.json")
submission_zip = os.path.join(output_dir, "submission.zip")
os.makedirs(output_dir, exist_ok=True)

train_ratio = 0.4

max_sent_tokens = 64
max_chunk_tokens = 600
min_chunk_tokens = 150
shift_percentile = 25
window_size = 2

max_enc_len = 640
chunk_summary_len = 80
doc_summary_len = 160
num_beams = 1
summary_batch_size = 16
doc_batch_size = 8
max_reduce_steps = 4
embed_target = "final"

max_doc_tokens = 512
init_rank = 24
target_rank = 16
lora_alpha = 24
lora_dropout = 0.05
target_modules = ("q_proj", "k_proj", "v_proj", "out_proj")
adapt_ffn = True
warmup_ratio = 0.15
freeze_ratio = 0.30
beta = 0.85
orth_weight = 0.5

temperature = 0.05
learning_rate = 5e-4
weight_decay = 0.01
num_epochs = 5
warmup_share = 0.05
batch_size = 4
num_negatives = 3
neg_pool_size = 10
top_k = 7

if debug_mode:
    num_epochs = 1
    train_ratio = 1.0
    debug_num_queries = 6
    debug_pool_size = 40

print("settings loaded | quick debug run:", debug_mode, "| train fraction:", train_ratio)


settings loaded | quick debug run: False | train fraction: 0.4


In [6]:
from model_arch import EncoderDecoderLM, LMConfig

vocab_file = hf_hub_download(tokenizer_repo, "vocab.json", token=hf_token)
merges_file = hf_hub_download(tokenizer_repo, "merges.txt", token=hf_token)
tokenizer = MyCustomTokenizer(vocab_file=vocab_file, merges_file=merges_file)
pad_token_id = tokenizer.pad_token_id
eos_token_id = tokenizer.eos_token_id

model_config = LMConfig(
    vocab_size=len(tokenizer),
    max_seq_len=context_length,
    d_model=hidden_size,
    n_layers=num_layers,
    n_heads=num_heads,
    d_ff=4 * hidden_size,
    dropout=0,
    bos_token_id=tokenizer.bos_token_id,
)

def load_model(repo_or_folder):
    if os.path.isdir(repo_or_folder):
        weights_file = os.path.join(repo_or_folder, "pytorch_model.bin")
    else:
        weights_file = hf_hub_download(repo_or_folder, "pytorch_model.bin", token=hf_token)
    model = EncoderDecoderLM(model_config)
    model.load_state_dict(torch.load(weights_file, map_location="cpu"))
    return model.to(device).eval()

base_model = load_model(base_model_repo)
for param in base_model.parameters():
    param.requires_grad = False

print(f"Phase-1 model loaded: {sum(p.numel() for p in base_model.parameters()) / 1e6:.1f}M parameters")


vocab.json:   0%|          | 0.00/640k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/320k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  489MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Phase-1 model loaded: 122.2M parameters


In [7]:
api = HfApi()
hub_files = api.list_repo_files(dataset_repo, repo_type="dataset", token=hf_token)
dataset_files = [f for f in hub_files if f.startswith("Phase_7_PCR/")]
print("\n".join(dataset_files))

def get_dataset_file(file_name):
    matches = [f for f in dataset_files if f.endswith("/" + file_name)]
    assert matches, f"{file_name} not found among {dataset_files}"
    return hf_hub_download(dataset_repo, matches[0], repo_type="dataset", token=hf_token)

id_field_names = ("id", "case_id", "query_id", "doc_id", "candidate_id")

def split_sentences(text):
    if isinstance(text, str):
        text = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in text if isinstance(s, str) and s.strip()]

def parse_cases(raw_data):
    if isinstance(raw_data, dict):
        entries = list(raw_data.items())
    else:
        entries = [(next((rec[k] for k in id_field_names if k in rec), None), rec) for rec in raw_data]
    parsed = {}
    for case_id, rec in entries:
        if isinstance(rec, dict):
            text = rec.get("text", rec.get("document", ""))
            relevant = rec.get("relevant_candidates", [])
        else:
            text = rec
            relevant = []
        parsed[str(case_id)] = {
            "sentences": split_sentences(text),
            "relevant": [str(x) for x in relevant],
        }
    return parsed

def load_split(query_file, pool_file):
    raw_queries = json.load(open(get_dataset_file(query_file), encoding="utf-8"))
    raw_pool = json.load(open(get_dataset_file(pool_file), encoding="utf-8"))
    first_item = str(list(raw_queries.items())[0])[:200] if isinstance(raw_queries, dict) else str(raw_queries[0])[:200]
    print(f"{query_file}: {type(raw_queries).__name__}, first record -> {first_item}")
    return parse_cases(raw_queries), parse_cases(raw_pool)

data_splits = {}
for split_name, (q_file, p_file) in {
    "train": ("train.json", "train_candidates.json"),
    "dev": ("dev.json", "dev_candidates.json"),
    "test": ("test.json", "test_candidates.json"),
}.items():
    queries, pool = load_split(q_file, p_file)
    avg_rel = np.mean([len(q["relevant"]) for q in queries.values()])
    print(f"{split_name}: {len(queries)} queries | {len(pool)} candidates | {avg_rel:.2f} relevant per query")
    data_splits[split_name] = (queries, pool)

def make_debug_subset(queries, pool, num_queries, pool_size):
    chosen = list(queries)[:num_queries]
    keep = []
    for q in chosen:
        keep += [c for c in queries[q]["relevant"] if c in pool and c not in keep]
    for c in pool:
        if len(keep) >= pool_size:
            break
        if c not in keep:
            keep.append(c)
    return {q: queries[q] for q in chosen}, {c: pool[c] for c in keep}

if debug_mode:
    data_splits = {
        name: make_debug_subset(q, p, debug_num_queries, debug_pool_size)
        for name, (q, p) in data_splits.items()
    }
    print("debug sizes:", {name: (len(q), len(p)) for name, (q, p) in data_splits.items()})


Phase_7_PCR/Dev/dev.json
Phase_7_PCR/Dev/dev_candidates.json
Phase_7_PCR/Test/test.json
Phase_7_PCR/Test/test_candidates.json
Phase_7_PCR/Train/train.json
Phase_7_PCR/Train/train_candidates.json


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train.json: dict, first record -> ('0000000599', {'text': ['PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P
train: 827 queries | 4320 candidates | 6.42 relevant per query


... (4 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test.json: dict, first record -> ('0000021652', {'text': ["PETITIONER: <NAME> Vs. RESPONDENT: <NAME> & ORS. DATE OF JUDGMENT26/09/1989 BENCH: <NAME>, <NAME> (J) BENCH: <NAME>, <NAME> (J) RAY, <NAME> (J) CITATION: 1990 AIR 209 1989 SC
test: 237 queries | 1727 candidates | 0.00 relevant per query


In [8]:
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def pad_batch(token_lists):
    width = max(1, max(len(t) for t in token_lists))
    ids = torch.full((len(token_lists), width), pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(token_lists), width), dtype=torch.long)
    for row, toks in enumerate(token_lists):
        if toks:
            ids[row, width - len(toks):] = torch.tensor(toks, dtype=torch.long)
            mask[row, width - len(toks):] = 1
    return ids.to(device), mask.to(device)

def mean_pooling(hidden_states, mask):
    weights = mask.unsqueeze(-1).to(hidden_states.dtype)
    return (hidden_states * weights).sum(1) / weights.sum(1).clamp(min=1)

@torch.no_grad()
def encode_batch(vector_fn, token_lists, batch_size=64, max_len=512):
    token_lists = [t[:max_len] for t in token_lists]
    order = sorted(range(len(token_lists)), key=lambda i: len(token_lists[i]))
    outputs = []
    for start in range(0, len(order), batch_size):
        batch_indices = order[start:start + batch_size]
        ids, mask = pad_batch([token_lists[i] for i in batch_indices])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            vecs = vector_fn(ids, mask)
        outputs.append(F.normalize(vecs.float(), dim=-1).cpu())
    sorted_vecs = torch.cat(outputs) if outputs else torch.zeros(0, model_config.d_model)
    final_vecs = torch.empty_like(sorted_vecs)
    final_vecs[torch.tensor(order, dtype=torch.long)] = sorted_vecs
    return final_vecs

def encode_sentence(ids, mask):
    return mean_pooling(base_model.encode(ids, mask), mask)

def find_topic_chunks(sentence_vectors, sentence_lengths):
    n = len(sentence_lengths)
    if n == 0:
        return []
    if n == 1:
        return [(0, 1)]
    w = window_size

    gap_similarity = np.empty(n - 1, dtype=np.float32)
    for g in range(n - 1):
        before = sentence_vectors[max(0, g - w + 1):g + 1].mean(0)
        after = sentence_vectors[g + 1:g + 1 + w].mean(0)
        gap_similarity[g] = float(before @ after / (np.linalg.norm(before) * np.linalg.norm(after) + 1e-08))
    dip_threshold = np.percentile(gap_similarity, shift_percentile)

    spans = []
    chunk_start = 0
    chunk_len = 0
    for i in range(n):
        if chunk_len > 0 and chunk_len + sentence_lengths[i] > max_chunk_tokens:
            spans.append((chunk_start, i))
            chunk_start = i
            chunk_len = 0
        chunk_len += sentence_lengths[i]
        is_low_dip = gap_similarity[i] < dip_threshold if i < n - 1 else False
        is_deepest_nearby = (
            i < n - 1 and gap_similarity[i] <= gap_similarity[max(0, i - w):i + w + 1].min()
        )
        if i < n - 1 and chunk_len >= min_chunk_tokens and is_low_dip and is_deepest_nearby:
            spans.append((chunk_start, i + 1))
            chunk_start = i + 1
            chunk_len = 0
    if chunk_start < n:
        spans.append((chunk_start, n))

    if len(spans) > 1:
        last_len = sum(sentence_lengths[spans[-1][0]:spans[-1][1]])
        prev_len = sum(sentence_lengths[spans[-2][0]:spans[-2][1]])
        if last_len < min_chunk_tokens // 2 and last_len + prev_len <= max_chunk_tokens:
            spans[-2:] = [(spans[-2][0], spans[-1][1])]
    return spans


In [9]:
def extract_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

class CentroidSummarizer:

    def summarise(self, texts, token_budget):
        results = []
        for text in texts:
            sentences = extract_sentences(text)
            if not sentences:
                results.append("")
                continue
            token_lists = [tokenizer.encode(s, add_special_tokens=False) for s in sentences]
            vecs = encode_batch(encode_sentence, [t[:max_sent_tokens] for t in token_lists])
            centre = F.normalize(vecs.mean(0, keepdim=True), dim=-1)
            rankings = torch.argsort(-(vecs @ centre.T).squeeze(1)).tolist()
            picked = []
            used_tokens = 0
            for i in rankings:
                if used_tokens + len(token_lists[i]) > token_budget and picked:
                    continue
                picked.append(i)
                used_tokens += len(token_lists[i])
                if used_tokens >= token_budget:
                    break
            results.append(" ".join(sentences[i] for i in sorted(picked)))
        return results

class AbstractiveSummarizer:

    def __init__(self, model, beams=1, batch_size=16):
        self.model = model
        self.beams = beams
        self.batch_size = batch_size
        self.banned_tokens = sorted({
            t for t in (pad_token_id, eos_token_id, tokenizer.unk_token_id, tokenizer.bos_token_id)
            if t is not None
        })

    @torch.no_grad()
    def summarise(self, texts, token_budget):
        token_lists = [tokenizer.encode(t, add_special_tokens=False)[:max_enc_len] for t in texts]
        order = sorted(range(len(texts)), key=lambda i: len(token_lists[i]))
        summaries = [""] * len(texts)
        for start in range(0, len(order), self.batch_size):
            batch_indices = order[start:start + self.batch_size]
            batch = [token_lists[i] for i in batch_indices]
            ids, mask = pad_batch(batch)
            new_tokens = min(token_budget, max(24, int(0.5 * min(len(b) for b in batch))))
            if self.beams > 1:
                out = self.model.generate_beam(
                    idx=ids,
                    max_new_tokens=new_tokens,
                    num_beams=self.beams,
                    temperature=0.3,
                    attention_mask=mask,
                    no_repeat_ngram_size=3,
                    banned_token_ids=self.banned_tokens,
                )
            else:
                out = self.model.generate(
                    ids,
                    max_new_tokens=new_tokens,
                    temperature=0.1,
                    top_p=1.0,
                    eos_token_id=eos_token_id,
                    attention_mask=mask,
                )
            for row, gen in zip(batch_indices, out[:, ids.size(1):].tolist()):
                if eos_token_id in gen:
                    gen = gen[:gen.index(eos_token_id)]
                summaries[row] = tokenizer.decode(gen, skip_special_tokens=True)
        return summaries

if summarizer_repo:
    summarizer = AbstractiveSummarizer(load_model(summarizer_repo), num_beams, summary_batch_size)
    print("summary writer: Phase-6 model", summarizer_repo)
else:
    summarizer = CentroidSummarizer()
    print("summary writer: EXTRACTIVE STAND-IN. Set phase6_summarizer_repo for real abstractive summaries.")


summary writer: EXTRACTIVE STAND-IN. Set phase6_summarizer_repo for real abstractive summaries.


In [10]:
def chunk_texts_by_budget(pieces, token_limit):
    bundles = []
    current = []
    current_len = 0
    for piece in pieces:
        n = count_tokens(piece)
        if current and current_len + n > token_limit:
            bundles.append(" ".join(current))
            current = []
            current_len = 0
        current.append(piece)
        current_len += n
    if current:
        bundles.append(" ".join(current))
    return bundles

def load_cached_summaries(path):
    cache = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                cache[record["id"]] = record
    return cache

def generate_hierarchical_summaries(sentences_by_case, save_path):
    saved = load_cached_summaries(save_path)
    pending = [c for c in sentences_by_case if c not in saved]
    print(f"{len(sentences_by_case)} cases | {len(saved)} already done | {len(pending)} to do")
    start_time = time.time()
    with open(save_path, "a", encoding="utf-8") as out_file:
        for first in range(0, len(pending), doc_batch_size):
            round_cases = pending[first:first + doc_batch_size]

            chunk_texts = []
            chunk_owner = []
            for case_id in round_cases:
                sentences = sentences_by_case[case_id]
                if not sentences:
                    continue
                token_lists = [tokenizer.encode(s, add_special_tokens=False) for s in sentences]
                sentence_vecs = encode_batch(
                    encode_sentence, [t[:max_sent_tokens] for t in token_lists]
                ).numpy()
                for (a, b) in find_topic_chunks(sentence_vecs, [len(t) for t in token_lists]):
                    chunk_texts.append(" ".join(sentences[a:b]))
                    chunk_owner.append(case_id)

            part_summaries = defaultdict(list)
            for case_id, summary in zip(chunk_owner, summarizer.summarise(chunk_texts, chunk_summary_len)):
                part_summaries[case_id].append(summary)
            chunk_counts = {c: len(part_summaries[c]) for c in round_cases}

            for _ in range(max_reduce_steps):
                too_long = {
                    c: chunk_texts_by_budget(part_summaries[c], max_enc_len)
                    for c in round_cases
                    if part_summaries[c] and count_tokens(" ".join(part_summaries[c])) > max_enc_len
                }
                if not too_long:
                    break
                all_bundles = [b for c in too_long for b in too_long[c]]
                shrunk = summarizer.summarise(all_bundles, chunk_summary_len)
                cursor = 0
                for c in too_long:
                    part_summaries[c] = shrunk[cursor:cursor + len(too_long[c])]
                    cursor += len(too_long[c])

            glued = {c: " ".join(part_summaries[c]) for c in round_cases}

            nonempty = [c for c in round_cases if glued[c].strip()]
            final_summaries = dict(
                zip(nonempty, summarizer.summarise([glued[c] for c in nonempty], doc_summary_len))
            )

            for c in round_cases:
                record = {
                    "id": c,
                    "parts": part_summaries[c],
                    "glued": glued[c],
                    "final": final_summaries.get(c, ""),
                    "n_chunks": chunk_counts[c],
                }
                saved[c] = record
                out_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_file.flush()
            print(f"  finished {min(first + doc_batch_size, len(pending))}/{len(pending)} | {time.time() - start_time:.0f}s elapsed")
    return {c: saved[c] for c in sentences_by_case}

def get_text_for_embedding(record):
    text = record.get(embed_target) or record.get("glued") or ""
    return text if text.strip() else " ".join(record.get("parts", []))


In [11]:
sample_queries, _ = data_splits["dev"]
sample_id = next(iter(sample_queries))
sample_sentences = sample_queries[sample_id]["sentences"]
sample_result = generate_hierarchical_summaries(
    {sample_id: sample_sentences}, os.path.join(output_dir, "walkthrough_cache.jsonl")
)[sample_id]

orig_tokens = sum(count_tokens(s) for s in sample_sentences)
print(f"\ncase {sample_id}: {len(sample_sentences)} sentences, {orig_tokens} tokens")
print(f"cut into {sample_result['n_chunks']} topic chunks")
for k, part in enumerate(sample_result["parts"][:3]):
    print(f"  chunk-{k + 1} summary: {part[:200]}")
print(f"glued length: {count_tokens(sample_result['glued'])} tokens")
print(f"final fingerprint: {count_tokens(sample_result['final'])} tokens  (compression x{orig_tokens / max(1, count_tokens(sample_result['final'])):.0f})")
print("final text:", sample_result["final"][:400])


1 cases | 0 already done | 1 to do
  finished 1/1 | 1s elapsed

case 0000012637: 71 sentences, 6724 tokens
cut into 16 topic chunks
  chunk-1 summary: In the face of the statement so recorded by the Income-tax Officer, the <ORG> does not appear to have been justified in relying upon inferences in ascertaining whether the earlier borrowal was on capi
  chunk-2 summary: Expenses incurred in raising capital are expenses of exactly the same character whether the capital is raised at the flotation of the company or thereafter : <ORG> v. Although the conclusion of <ORG> 
  chunk-3 summary: To conclude we hold that the expenditure of Rs. The answer to the question referred, therefore, must be in the affirmative. The appeal is allowed, the judgment of <ORG> set aside and the question refe
glued length: 221 tokens
final fingerprint: 159 tokens  (compression x42)
final text: In the face of the statement so recorded by the Income-tax Officer, the <ORG> does not appear to have been justified in re

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_full_text(case):
    return " ".join(case["sentences"])

def compute_tfidf_similarity(queries, pool, query_ids, pool_ids):
    vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.6, max_features=300000, dtype=np.float32)
    pool_matrix = vectorizer.fit_transform([get_full_text(pool[i]) for i in pool_ids])
    query_matrix = vectorizer.transform([get_full_text(queries[i]) for i in query_ids])
    return (query_matrix @ pool_matrix.T).toarray()

train_queries, train_pool = data_splits["train"]
valid_train_ids = [q for q in train_queries if any(c in train_pool for c in train_queries[q]["relevant"])]
random.Random(seed).shuffle(valid_train_ids)
num_train = max(1, int(train_ratio * len(valid_train_ids)))
train_ids = valid_train_ids[:num_train]
print(f"usable training queries: {len(valid_train_ids)} -> fine-tuning on {num_train} ({100 * num_train / len(valid_train_ids):.0f} %)")

train_pool_ids = list(train_pool)
train_tfidf_sim = compute_tfidf_similarity(train_queries, train_pool, train_ids, train_pool_ids)

training_pairs = []
needed_case_ids = set()
for row, q in enumerate(train_ids):
    relevant = set(c for c in train_queries[q]["relevant"] if c in train_pool)
    sorted_indices = np.argsort(-train_tfidf_sim[row])
    hard_negs = [
        train_pool_ids[j]
        for j in sorted_indices
        if train_pool_ids[j] not in relevant and train_pool_ids[j] != q
    ][:neg_pool_size]
    training_pairs.append({
        "query": q,
        "positives": sorted(relevant),
        "negatives": hard_negs,
        "relevant_set": relevant,
    })
    needed_case_ids |= {q} | relevant | set(hard_negs)
print(f"cases to summarise for training: {len(needed_case_ids)}")

def get_sentences_by_ids(queries, pool, ids):
    return {i: (queries[i] if i in queries else pool[i])["sentences"] for i in ids}

case_summaries = {}
case_summaries["train"] = generate_hierarchical_summaries(
    get_sentences_by_ids(train_queries, train_pool, needed_case_ids),
    os.path.join(output_dir, "fingerprints_train.jsonl"),
)

for split_name in ("dev", "test"):
    queries, pool = data_splits[split_name]
    all_case_ids = list(dict.fromkeys(list(queries) + list(pool)))
    case_summaries[split_name] = generate_hierarchical_summaries(
        get_sentences_by_ids(queries, pool, all_case_ids),
        os.path.join(output_dir, f"fingerprints_{split_name}.jsonl"),
    )

case_tokens = {
    split: {
        i: tokenizer.encode(get_text_for_embedding(rec), add_special_tokens=False)[:max_doc_tokens]
        for i, rec in case_summaries[split].items()
    }
    for split in case_summaries
}


usable training queries: 817 -> fine-tuning on 326 (40 %)
cases to summarise for training: 2923
2923 cases | 0 already done | 2923 to do
  finished 8/2923 | 2s elapsed
  finished 16/2923 | 4s elapsed
  finished 24/2923 | 6s elapsed
  finished 32/2923 | 8s elapsed
  finished 40/2923 | 11s elapsed
  finished 48/2923 | 15s elapsed
  finished 56/2923 | 17s elapsed
  finished 64/2923 | 19s elapsed
  finished 72/2923 | 23s elapsed
  finished 80/2923 | 25s elapsed
  finished 88/2923 | 29s elapsed
  finished 96/2923 | 30s elapsed
  finished 104/2923 | 33s elapsed
  finished 112/2923 | 36s elapsed
  finished 120/2923 | 40s elapsed
  finished 128/2923 | 42s elapsed
  finished 136/2923 | 48s elapsed
  finished 144/2923 | 49s elapsed
  finished 152/2923 | 51s elapsed
  finished 160/2923 | 54s elapsed
  finished 168/2923 | 55s elapsed
  finished 176/2923 | 59s elapsed
  finished 184/2923 | 63s elapsed
  finished 192/2923 | 67s elapsed
  finished 200/2923 | 75s elapsed
  finished 208/2923 | 79s elap

In [13]:
class AdaLoRALinear(nn.Module):

    def __init__(self, frozen_linear: nn.Linear, init_rank=24, alpha=24, dropout=0.0):
        super().__init__()
        self.frozen = frozen_linear
        for param in self.frozen.parameters():
            param.requires_grad = False
        device = frozen_linear.weight.device
        self.init_rank = init_rank
        self.scaling = alpha / init_rank
        self.lam = nn.Parameter(torch.zeros(init_rank, device=device))
        self.left = nn.Parameter(torch.randn(frozen_linear.out_features, init_rank, device=device) * 0.02)
        self.right = nn.Parameter(torch.randn(init_rank, frozen_linear.in_features, device=device) * 0.02)
        self.register_buffer("keep", torch.ones(init_rank, device=device))
        self.dropout = nn.Dropout(dropout)
        for name in ("lam", "left", "right"):
            for stat in ("sens", "unc"):
                self.register_buffer(f"{stat}_{name}", torch.zeros_like(getattr(self, name)), persistent=False)

    def forward(self, x):
        mid = F.linear(self.dropout(x), self.right) * (self.lam * self.keep)
        return self.frozen(x) + self.scaling * F.linear(mid, self.left)

    @torch.no_grad()
    def update_importance(self, beta=0.85):
        for name in ("lam", "left", "right"):
            p = getattr(self, name)
            if p.grad is None:
                continue
            ipt = (p * p.grad).abs()
            sens = getattr(self, f"sens_{name}")
            unc = getattr(self, f"unc_{name}")
            sens.mul_(beta).add_(ipt, alpha=1 - beta)
            unc.mul_(beta).add_((ipt - sens).abs(), alpha=1 - beta)

    @torch.no_grad()
    def triplet_scores(self):
        s = {name: getattr(self, f"sens_{name}") * getattr(self, f"unc_{name}") for name in ("lam", "left", "right")}
        return s["lam"] + s["left"].mean(dim=0) + s["right"].mean(dim=1)

    @torch.no_grad()
    def effective_rank(self):
        return int(((self.lam * self.keep) != 0).sum())

def apply_adalora(model, attention_mats, init_rank, alpha, dropout, include_ffn=True):
    for param in model.parameters():
        param.requires_grad = False
    for block in model.enc_blocks:
        for mat_name in attention_mats:
            setattr(block.attn, mat_name, AdaLoRALinear(getattr(block.attn, mat_name), init_rank, alpha, dropout))
        if include_ffn:
            block.ffn.net[0] = AdaLoRALinear(block.ffn.net[0], init_rank, alpha, dropout)
            block.ffn.net[3] = AdaLoRALinear(block.ffn.net[3], init_rank, alpha, dropout)
    return model

def compute_orth_penalty(layers):
    total = 0.0
    for layer in layers:
        eye = torch.eye(layer.init_rank, device=layer.left.device)
        total = total + torch.norm(layer.left.T @ layer.left - eye) + torch.norm(layer.right @ layer.right.T - eye)
    return total / len(layers)

@torch.no_grad()
def select_top_triplets(layers, budget):
    scores = torch.cat([layer.triplet_scores() for layer in layers])
    chosen = torch.zeros_like(scores, dtype=torch.bool)
    chosen[torch.topk(scores, max(1, min(int(budget), scores.numel()))).indices] = True
    return list(chosen.split([layer.init_rank for layer in layers]))

def print_rank_distribution(model):
    by_type = {}
    by_block = {}
    for name, layer in model.named_modules():
        if isinstance(layer, AdaLoRALinear):
            parts = name.split(".")
            kind = parts[-1] if parts[-2] == "attn" else f"ffn.{parts[-1]}"
            by_type.setdefault(kind, []).append(layer.effective_rank())
            by_block.setdefault(int(parts[parts.index("enc_blocks") + 1]), []).append(layer.effective_rank())
    print("average rank kept per matrix type :", {k: round(float(np.mean(v)), 1) for k, v in by_type.items()})
    print("average rank kept per encoder block:", {k: round(float(np.mean(v)), 1) for k, v in sorted(by_block.items())})

class Retriever(nn.Module):

    def __init__(self, encoder_model):
        super().__init__()
        self.encoder = encoder_model

    def forward(self, ids, mask):
        return F.normalize(mean_pooling(self.encoder.encode(ids, mask), mask).float(), dim=-1)

retriever = Retriever(
    apply_adalora(
        copy.deepcopy(base_model),
        target_modules,
        init_rank,
        lora_alpha,
        lora_dropout,
        adapt_ffn,
    )
).to(device)

adapter_layers = [m for m in retriever.modules() if isinstance(m, AdaLoRALinear)]
trainable_count = sum(p.numel() for p in retriever.parameters() if p.requires_grad)
total_count = sum(p.numel() for p in retriever.parameters())

print(f"AdaLoRA: {len(adapter_layers)} adapted matrices, start rank {init_rank} -> target average rank {target_rank} ({len(adapter_layers) * init_rank} -> {len(adapter_layers) * target_rank} triplets)")
print(f"         training {trainable_count / 1e6:.2f}M of {total_count / 1e6:.1f}M weights ({100 * trainable_count / total_count:.2f} %) at the start")


AdaLoRA: 36 adapted matrices, start rank 24 -> target average rank 16 (864 -> 576 triplets)
         training 1.99M of 124.2M weights (1.60 %) at the start


In [14]:
def compute_metrics(predicted, ground_truth, k=top_k):
    hits = sum(len(set(predicted[q][:k]) & set(ground_truth[q])) for q in ground_truth)
    total_returned = sum(len(predicted[q][:k]) for q in ground_truth)
    total_relevant = sum(len(set(ground_truth[q])) for q in ground_truth)
    precision = hits / max(total_returned, 1)
    recall = hits / max(total_relevant, 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1

def get_top_k(score_matrix, query_ids, pool_ids, k=top_k):
    pool_idx_map = {p: j for j, p in enumerate(pool_ids)}
    picks = {}
    for row, q in enumerate(query_ids):
        scores = score_matrix[row].copy()
        if q in pool_idx_map:
            scores[pool_idx_map[q]] = -np.inf
        picks[q] = [pool_ids[j] for j in np.argsort(-scores)[:k]]
    return picks

@torch.no_grad()
def embed_cases(model, split_name, case_ids):
    model.eval()
    return encode_batch(
        model,
        [case_tokens[split_name][i] for i in case_ids],
        batch_size=32,
        max_len=max_doc_tokens,
    ).numpy()

dev_queries, dev_pool = data_splits["dev"]
dev_qids = list(dev_queries)
dev_pool_ids = list(dev_pool)
dev_ground_truth = {q: dev_queries[q]["relevant"] for q in dev_qids}
dev_tfidf_sim = compute_tfidf_similarity(dev_queries, dev_pool, dev_qids, dev_pool_ids)

def eval_dev(keyword_weight=0.0):
    sims = embed_cases(retriever, "dev", dev_qids) @ embed_cases(retriever, "dev", dev_pool_ids).T
    return compute_metrics(get_top_k(sims + keyword_weight * dev_tfidf_sim, dev_qids, dev_pool_ids), dev_ground_truth)

p, r, f = compute_metrics(get_top_k(dev_tfidf_sim, dev_qids, dev_pool_ids), dev_ground_truth)
print(f"[reference] keyword matching (TF-IDF) on raw text        : precision {p:.4f}  recall {r:.4f}  F1@7 {f:.4f}")

p, r, f = eval_dev()
print(f"[reference] Phase-1 model before any fine-tuning          : precision {p:.4f}  recall {r:.4f}  F1@7 {f:.4f}")


[reference] keyword matching (TF-IDF) on raw text        : precision 0.3862  recall 0.3279  F1@7 0.3546
[reference] Phase-1 model before any fine-tuning          : precision 0.1065  recall 0.0904  F1@7 0.0978


In [15]:
query_relevance = {ex["query"]: ex["relevant_set"] for ex in training_pairs}
trainable_params = [p for p in retriever.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

steps_per_epoch = max(1, math.ceil(len(training_pairs) / batch_size))
total_steps = steps_per_epoch * num_epochs
warmup_steps = max(1, int(warmup_share * total_steps))

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps)))

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
grad_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

budget_start = len(adapter_layers) * init_rank
budget_end = min(budget_start, len(adapter_layers) * target_rank)
warmup_end_step = int(warmup_ratio * total_steps)
freeze_step = max(warmup_end_step + 1, total_steps - int(freeze_ratio * total_steps))
ranks_frozen = False

def get_budget(step):
    if step <= warmup_end_step:
        return budget_start
    if step >= freeze_step:
        return budget_end
    progress = (step - warmup_end_step) / (freeze_step - warmup_end_step)
    return budget_end + (budget_start - budget_end) * (1 - progress) ** 3

def count_active_triplets():
    return sum(layer.effective_rank() for layer in adapter_layers)

print(f"[AdaLoRA] budget {budget_start} -> {budget_end} triplets | warm-up until step {warmup_end_step} | pattern frozen from step {freeze_step} of {total_steps}")

def save_adapter_state():
    state = {name: p.detach().cpu().clone() for name, p in retriever.named_parameters() if p.requires_grad}
    state.update({name: b.detach().cpu().clone() for name, b in retriever.named_buffers() if name.endswith(".keep")})
    return state

best_f1 = -1.0
best_adapter_state = save_adapter_state()
train_history = []
global_step = 0
rng = random.Random(seed)

for epoch in range(num_epochs):
    retriever.train()
    order = list(range(len(training_pairs)))
    rng.shuffle(order)

    for first in range(0, len(order), batch_size):
        batch = [training_pairs[i] for i in order[first:first + batch_size]]
        n = len(batch)
        batch_qids = [ex["query"] for ex in batch]
        pos_ids = [rng.choice(ex["positives"]) for ex in batch]
        neg_ids = [neg for ex in batch for neg in rng.sample(ex["negatives"], min(num_negatives, len(ex["negatives"])))]
        cand_ids_batch = pos_ids + neg_ids

        ids, mask = pad_batch([case_tokens["train"][i] for i in batch_qids + cand_ids_batch])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            vectors = retriever(ids, mask)

        q_vecs = vectors[:n]
        a_vecs = vectors[n:]
        scores = (q_vecs @ a_vecs.T) / temperature

        other_positives = torch.zeros_like(scores, dtype=torch.bool)
        for row, q in enumerate(batch_qids):
            for col, answer in enumerate(cand_ids_batch):
                if col != row and answer in query_relevance[q]:
                    other_positives[row, col] = True

        retrieval_loss = F.cross_entropy(scores.masked_fill(other_positives, -1e4), torch.arange(n, device=device))
        ortho_loss = compute_orth_penalty(adapter_layers)
        loss = retrieval_loss + orth_weight * ortho_loss

        optimizer.zero_grad(set_to_none=True)
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)

        if not ranks_frozen:
            if bool(torch.isfinite(torch.stack([p.grad.abs().max() for p in trainable_params if p.grad is not None])).all()):
                for layer in adapter_layers:
                    layer.update_importance(beta)

        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        grad_scaler.step(optimizer)
        grad_scaler.update()
        lr_scheduler.step()
        global_step += 1

        if not ranks_frozen and global_step > warmup_end_step:
            triplet_masks = select_top_triplets(adapter_layers, get_budget(global_step))
            with torch.no_grad():
                for layer, mask_t in zip(adapter_layers, triplet_masks):
                    layer.lam.masked_fill_(~mask_t, 0.0)
                    if global_step >= freeze_step:
                        layer.keep.copy_(mask_t.float())
            ranks_frozen = global_step >= freeze_step

        if global_step == 1 or global_step % 10 == 0:
            print(f"epoch {epoch + 1}/{num_epochs} | step {global_step}/{total_steps} | loss {loss.item():.4f} "
                  f"(contrastive {retrieval_loss.item():.4f}, orthogonality {ortho_loss.item():.4f}) | triplets kept {count_active_triplets()}/{budget_start}")

    p, r, f = eval_dev()
    train_history.append({"epoch": epoch + 1, "precision": p, "recall": r, "f1_at_7": f})
    print(f"== after epoch {epoch + 1}: dev precision {p:.4f} | recall {r:.4f} | F1@7 {f:.4f} | triplets kept {count_active_triplets()}/{budget_start}")

    if f > best_f1:
        best_f1 = f
        best_adapter_state = save_adapter_state()

retriever.load_state_dict({**retriever.state_dict(), **best_adapter_state})
adapter_path = os.path.join(output_dir, f"adalora_r{init_rank}to{target_rank}_{int(train_ratio * 100)}pct.pt")
torch.save(best_adapter_state, adapter_path)
print("best dev F1@7:", round(best_f1, 4), "| adapters saved to", adapter_path)
print_rank_distribution(retriever)


/tmp/ipykernel_22/3731093555.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  loss_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


[AdaLoRA] budget 864 -> 576 triplets | warm-up until step 61 | pattern frozen from step 287 of 410
epoch 1/5 | step 1/410 | loss 5.7025 (contrastive 2.6594, orthogonality 6.0863) | triplets kept 864/864
epoch 1/5 | step 10/410 | loss 5.5845 (contrastive 2.7450, orthogonality 5.6789) | triplets kept 864/864
epoch 1/5 | step 20/410 | loss 5.0331 (contrastive 2.6850, orthogonality 4.6962) | triplets kept 864/864
epoch 1/5 | step 30/410 | loss 4.2086 (contrastive 2.6770, orthogonality 3.0632) | triplets kept 864/864
epoch 1/5 | step 40/410 | loss 3.2070 (contrastive 2.6321, orthogonality 1.1499) | triplets kept 864/864
epoch 1/5 | step 50/410 | loss 2.9665 (contrastive 2.6924, orthogonality 0.5481) | triplets kept 864/864
epoch 1/5 | step 60/410 | loss 2.7500 (contrastive 2.6882, orthogonality 0.1235) | triplets kept 864/864
epoch 1/5 | step 70/410 | loss 2.7272 (contrastive 2.6821, orthogonality 0.0901) | triplets kept 830/864
epoch 1/5 | step 80/410 | loss 2.7325 (contrastive 2.6957, ort

In [16]:
dev_similarities = embed_cases(retriever, "dev", dev_qids) @ embed_cases(retriever, "dev", dev_pool_ids).T
weight_f1_scores = {
    w: compute_metrics(get_top_k(dev_similarities + w * dev_tfidf_sim, dev_qids, dev_pool_ids), dev_ground_truth)[2]
    for w in (0.0, 0.1, 0.25, 0.5, 1.0)
}
print("dev F1@7 for each keyword weight:", {w: round(v, 4) for w, v in weight_f1_scores.items()})
best_kw_weight = max(weight_f1_scores, key=weight_f1_scores.get)
print("using keyword weight", best_kw_weight)

test_queries, test_pool = data_splits["test"]
test_qids = list(test_queries)
test_pool_ids = list(test_pool)
test_scores = embed_cases(retriever, "test", test_qids) @ embed_cases(retriever, "test", test_pool_ids).T

if best_kw_weight > 0:
    test_scores = test_scores + best_kw_weight * compute_tfidf_similarity(test_queries, test_pool, test_qids, test_pool_ids)

predictions = get_top_k(test_scores, test_qids, test_pool_ids)

cand_set = set(test_pool_ids)
for q, picks in predictions.items():
    assert len(picks) == top_k and len(set(picks)) == top_k and all(c in cand_set for c in picks), q

if not debug_mode:
    assert set(predictions) == set(test_qids), "every official test query id must be present"

with open(pred_file, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(submission_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_file, arcname="prediction.json")

print(f"{len(predictions)} queries written -> {submission_zip}")


dev F1@7 for each keyword weight: {0.0: 0.0978, 0.1: 0.3369, 0.25: 0.358, 0.5: 0.3613, 1.0: 0.3558}
using keyword weight 0.5
237 queries written -> /kaggle/working/cs781_phase7/submission.zip
